# Reproduce Figure 1

Thin wrapper around `scripts/run_experiments.py`. Start with the offline `hashing` embedder to sanity-check the pipeline before spending time/API budget on `sentence_transformer` or `llm`.

In [ ]:
import sys
sys.path.insert(0, "../src")

from rme.data import LOADERS, train_test_split
from rme.embeddings import CachingEmbedder, HashingEmbedder
from rme.evaluation import bootstrap_pairwise_accuracy
from rme.models import MODEL_REGISTRY

list(LOADERS), list(MODEL_REGISTRY)

In [ ]:
dataset_name = "anthropic_hh_golden"  # needs `pip install rme[data]` + network
comparisons = LOADERS[dataset_name](max_examples=2000)
train, test = train_test_split(comparisons, test_frac=0.2, seed=0)
len(train), len(test)

In [ ]:
embedder = CachingEmbedder(HashingEmbedder())
results = {}
for name, cls in MODEL_REGISTRY.items():
    model = cls().fit(train, embedder)
    results[name] = bootstrap_pairwise_accuracy(model, test, n_runs=5)
results

For the full grid across datasets/embedders (Figure 1), use `scripts/run_experiments.py` directly -- it caches embeddings per (dataset, embedder) combo, which this notebook does not.